In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fontTools.unicodedata import script_name
%matplotlib inline
from sklearn.linear_model import LogisticRegressionCV
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from helpers.helper_functions import *

# Handling text 2 exercise

The Sheldon Cooper we all know and love (OK, some of us might not know him, and some might not love him) from the TV series "The Big Bang Theory" has gotten into an argument with Leonard from the same TV show. Sheldon insists that he knows the show better than anyone, and keeps making various claims about the show, which neither of them know how to prove or disprove. The two of them have reached out to you ladies and gentlemen, as data scientists, to help them. You will be given the full script of the series, with information on the episode, the scene, the person saying each dialogue line, and the dialogue lines themselves.

Leonard has challenged several of Sheldon's claims about the show, and throughout this exam you will see some of those and you will get to prove or disprove them, but remember: sometimes, we can neither prove a claim, nor disprove it!

## Task A: Picking up the shovel

**Note: You will use the data you preprocess in this task in all the subsequent ones.**

Our friends' argument concerns the entire show. We have given you a file in the `data/` folder that contains the script of every single episode. New episodes are indicated by '>>', new scenes by '>', and the rest of the lines are dialogue lines. Some lines are said by multiple people (for example, lines indicated by 'All' or 'Together'); **you must discard these lines**, for the sake of simplicity. However, you do not need to do it for Q1 in this task -- you'll take care of it when you solve Q2.

**Q1**. Your first task is to extract all lines of dialogue in each scene and episode, creating a dataframe where each row has the episode and scene where a dialogue line was said, the character who said it, and the line itself. You do not need to extract the proper name of the episode (e.g. episode 1 can appear as "Series 01 Episode 01 - Pilot Episode", and doesn't need to appear as "Pilot Episode"). Then, answer the following question: In total, how many scenes are there in each season? We're not asking about unique scenes; the same location appearing in two episodes counts as two scenes. You can use a Pandas dataframe with a season column and a scene count column as the response.

**Note: The data refers to seasons as "series".**

In [45]:
# your code goes here
import os, codecs, string, random

import re
import pandas as pd

dialogues = []
scenes = []

season = None
episode = None
scene = 0

with open("data/all_scripts.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        # New episode
        if line.startswith(">>"):
            episode = line[2:].strip()
            season = int(re.search(r"Series (\d+)", episode).group(1))
            scene = 0
        # New scene
        elif line.startswith(">"):
            scene += 1
            scenes.append({"season": season, "episode": episode, "scene": scene})
        # Dialogue
        else:
            character, text = line.split(":", 1)
            dialogues.append({
                "season": season,
                "episode": episode,
                "scene": scene,
                "character": character.strip(),
                "line": text.strip()
            })
df = pd.DataFrame(dialogues)

scenes_per_season = (
    df[["season", "episode", "scene"]]
    .drop_duplicates()
    .groupby("season")
    .size()
)
print(scenes_per_season)

season
1     159
2     231
3     236
4     279
5     254
6     304
7     332
8     327
9     337
10    346
dtype: int64


**Q2**. Now, let's define two sets of characters: all the characters, and recurrent characters. Recurrent characters are those who appear in more than one episode. For the subsequent sections, you will need to have a list of recurrent characters. Assume that there are no two _named characters_ (i.e. characters who have actual names and aren't referred to generically as "little girl", "grumpy grandpa", etc.) with the same name, i.e. there are no two Sheldons, etc. Generate a list of recurrent characters who have more than 90 dialogue lines in total, and then take a look at the list you have. If you've done this correctly, you should have a list of 20 names. However, one of these is clearly not a recurrent character. Manually remove that one, and print out your list of recurrent characters. To remove that character, pay attention to the _named character_ assumption we gave you earlier on. **For all the subsequent questions, you must only keep the dialogue lines said by the recurrent characters in your list.**

In [47]:
# your code goes here
# Remove multi-speaker dialogue
df = df[~df["character"].isin(["All", "Together"])]

# Characters appearing in more than one episode
episode_counts = df.groupby("character")["episode"].nunique()
recurrent = episode_counts[episode_counts > 1].index

# Recurrent characters with more than 90 lines
line_counts = df[df["character"].isin(recurrent)]["character"].value_counts()
recurrent_characters = line_counts[line_counts > 90].index.tolist()

print(len(recurrent_characters))
print(recurrent_characters)

# Manually remove the generic/non-character entry after inspecting the list
recurrent_characters.remove("Man")

# Only keep these characters from now on
df = df[df["character"].isin(recurrent_characters)]

20
['Sheldon', 'Leonard', 'Penny', 'Howard', 'Raj', 'Amy', 'Bernadette', 'Stuart', 'Priya', 'Mrs Cooper', 'Emily', 'Beverley', 'Mrs Wolowitz', 'Zack', 'Arthur', 'Wil', 'Leslie', 'Kripke', 'Man', 'Bert']


## Task B: Read the scripts carefully

### Part 1: Don't put the shovel down just yet

**Q3**. From each dialogue line, replace punctuation marks (listed in the EXCLUDE_CHARS variable provided in `helpers/helper_functions.py`) with whitespaces, and lowercase all the text. **Do not remove any stopwords, leave them be for all the questions in this task.**

In [49]:
# your code goes here
from helpers.helper_functions import EXCLUDE_CHARS

translation = str.maketrans({char: " " for char in EXCLUDE_CHARS})
df["line"] = df["line"].str.translate(translation).str.lower()
print(df.head())

   season                               episode  scene character  \
0       1  Series 01 Episode 01 – Pilot Episode      1   Sheldon   
1       1  Series 01 Episode 01 – Pilot Episode      1   Leonard   
2       1  Series 01 Episode 01 – Pilot Episode      1   Sheldon   
3       1  Series 01 Episode 01 – Pilot Episode      1   Leonard   
5       1  Series 01 Episode 01 – Pilot Episode      1   Leonard   

                                                line  
0  so if a photon is directed through a plane wit...  
1                         agreed  what s your point   
2  there s no point  i just think it s a good ide...  
3                                         excuse me   
5  one across is aegean  eight down is nabakov  t...  


**Q4**. For each term, calculate its "corpus frequency", i.e. its number of occurrences in the entire series. Visualize the distribution of corpus frequency using a histogram. Explain your observations. What are the appropriate x and y scales for this plot?

In [6]:
# your code goes here
terms = df["line"].str.split().explode()

corpus_frequency = terms.value_counts()

### Part 2: Talkativity
**Q5**. For each of the recurrent characters, calculate their total number of words uttered across all episodes. Based on this, who seems to be the most talkative character?

In [7]:
# your code goes here

## Task D: The Detective's Hat

Sheldon claims that given a dialogue line, he can, with an accuracy of above 70%, say whether it's by himself or by someone else. Leonard contests this claim, since he believes that this claimed accuracy is too high.

**Q6**. Divide the set of all dialogue lines into two subsets: the training set, consisting of all the seasons except the last two, and the test set, consisting of the last two seasons.

In [8]:
# your code goes here

**Q7**. Find the set of all words in the training set that are only uttered by Sheldon. Is it possible for Sheldon to identify himself only based on these? Use the test set to assess this possibility, and explain your method.

In [9]:
# your code goes here